# Primary Questions Analysis

## Primary Questions and Analysis Plan:

From the Analysis Framework, the following questions were suggested as the primary concern:

- What are great predictors of good sleep duration and sleep quality? 
- Are there patterns amongst individuals who have sleep disorders vs individuals who do not? 
- Is the type of sleep disorder reflected in the other variables/ can we differentiate between sleep disorders using the data provided?

Thus, each section will use various statistical methods to find the answers to these questions.

In [12]:
# setting up the notebook
import pandas as pd
from sklearn.model_selection import train_test_split

# data read in
df = pd.read_csv('./data/modified_sleep_health_and_lifestyle_dataset.csv')

## Section 1: What are the best predictors for sleep duration and quality?

This section will contain two parts: feature engineering and predictive modeling.

For feature engineering, both sleep duration and sleep quality have ranges for which they are contained within. Sleep quality is assumed to be on a scale from 1-10 and recorded as an integer. On the other hand, sleep duration being greater than 24 (hours/day) is not possible. One solution would be to turn these variables into binary (above some standard). A second option would be to scale the values to a 0-1 range (beta regression). Yet, another would be to merely keep them as is and interpret as necessary.

For this section, the first two methods will be used to both explore thresholds and retain as much information as possible. For the thresholds, sleep quality was set to 5 as being the middle ground. For sleep duration, the threshold was set to 7 as the average adult needs 7 hours of sleep. Additionally, the range for sleep duration will be assumed to be between 0-15 hours as 15 hours is the maximum number of hours an adult can naturally sleep for.

In [13]:
# sleep_quality to binary:
df['sleep_quality_binary'] = df['sleep_quality'].apply(lambda x: 1 if x >= 5 else 0)

# sleep duration to binary:
df['sleep_duration_binary'] = df['sleep_duration'].apply(lambda x: 1 if x >= 7 else 0)

# sleep quality rescaled to 0-1: 
df['sleep_quality_rescaled'] = (df['sleep_quality'] - 1)/9

# sleep duration rescaled to 0-1:
df['sleep_duration_rescaled'] = df['sleep_duration']/15


Before any regression, the dataset will first be split into train and test datasets with a 80/20 split.

In [14]:
# binary train test split:
X_quality = df.drop(columns=['sleep_quality','sleep_quality_binary', 'sleep_quality_rescaled'])
X_duration = df.drop(columns=['sleep_duration','sleep_duration_binary', 'sleep_duration_rescaled'])
y_quality_binary = df['sleep_quality_binary']
y_duration_binary = df['sleep_duration_binary']
X_train_qb, X_test_qb, y_train_qb, y_test_qb = train_test_split(X_quality, y_quality_binary, test_size=0.2, random_state=10)
X_train_db, X_test_db, y_train_db, y_test_db = train_test_split(X_duration, y_duration_binary, test_size=0.2, random_state=10)

# rescaled train test split:
y_quality_rescaled = df['sleep_quality_rescaled']
y_duration_rescaled = df['sleep_duration_rescaled']
X_train_qr, X_test_qr, y_train_qr, y_test_qr = train_test_split(X_quality, y_quality_rescaled, test_size=0.2, random_state=10)
X_train_dr, X_test_dr, y_train_dr, y_test_dr = train_test_split(X_duration, y_duration_rescaled, test_size=0.2, random_state=10)

### Rescaled Regression

This subsection will be for examining the rescaled variables and all conclusions derived. To start, fractional logistic regression and fractional probit regression will be run. My hypothesis is that due to the way these variables are typically distributed, the probit link function/ normal distribution assumption will work better to predict the values.

